In [1]:
# --- repo bootstrap: make src/ + this domain's config importable, run from repo root ---
# domains/credit_risk/ is mounted only in the credit-risk containers, so the
# domain-agnostic stages physically cannot import domain settings.
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
sys.path.insert(0, str(_ROOT / 'domains' / 'credit_risk'))
os.chdir(_ROOT)

<!-- # Aave V3.1 — model dataset (daily credit-risk panel)

Assembles the modeling table: the 2h protocol panel aggregated to daily grain
(events/flows sum, ratios REBUILT as ratio-of-sums from the daily flows — not
the mean of 12 bucket ratios), joined with the dense 24h liquidation +
user-account risk frames, restricted to the credit-risk feature set, with
forward-looking stress/volume targets. Exports `DF_model_dataset_24h.csv`. -->

In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import display

import adv_validation as adv
import domains.credit_risk.model_config as cfg
import model_dataset as mds
import transform as tf

MAIN = "DF_common_final_2"      # enriched 2h panel (61 cols); _1 is the 30-col base
LIQ, USER = "DF_liq_features_24h", "DF_user_features_24h"
MARKET = "DF_common_2"
TABLES = (MAIN, LIQ, USER, MARKET)
MAIN_COLS = 61                  # DF_common_final_2 width; guards the _1/_2 mix-up
PREVIEW_ROWS = 5

display(tf.output_timelines())

,batch,tables,first_date,last_date,select_with
0,2024-04 → 2025-03,14,2024-04-01,2025-03-31,'2024-04'
1,2025-04 → 2026-03,14,2025-04-01,2026-03-31,'2025-04'


In [3]:
# paste `select_with` value(s) from the table above; None = the newest saved batch
#   "2025-04"                 one batch
#   ("2025-04", "2026-03")    one batch by its full span
#   {"2024-04", "2025-04"}    several batches, JOINED in time order before any processing
#                             -> outputs land in model_data/ + target_data/batch_<first>_<last>/
BATCH = {"2024-04", "2025-04"}


In [4]:
run = tf.use_output_batch(BATCH)
# every model-stage export of this run lands in ONE folder keyed by the pinned timeline:
# model_data/batch_<start>_<end>/ (cfg.MODEL_DATA_DIR). Separate from transformed_data/,
# so §1's resolvers never see these files and a second window cannot overwrite this one.
MODEL_DIR = run.folder(cfg.MODEL_DATA_DIR)
print(run)
print(f" model-stage output -> {MODEL_DIR}/")

<CombinedRun 2024-04 → 2026-03 · joins 2024-04 → 2025-03 + 2025-04 → 2026-03 · 14 tables in every batch · 0 loaded · from transformed_data/>
 model-stage output -> model_data/batch_2024-04_2026-03/


In [5]:
MAIN = run.load(MAIN)     # enriched 2h panel (61 cols); _1 is the 30-col base
LIQ, USER = run.load(LIQ), run.load(USER)   # the model inputs
MARKET = run.load(MARKET)
SOURCES = {                     # loaded for reconciliation only, never joined
    "liq_2h": "df_liq_2h", "liq_24h": "df_liq_24h", "liq_7d": "df_liq_7d",
    "user_2h": "df_user_account_2h", "user_24h": "df_user_account_24h",
    "user_7d": "df_user_account_7d",
    "liq_feat_7d": "DF_liq_features_7d", "user_feat_7d": "DF_user_features_7d",
}
TABLES = (MAIN, LIQ, USER, *SOURCES.values())

# --- the 4 TARGET columns: forward-rolled in the targets section only, never a feature ---
# kept out of every df_* / DF_model_* frame, rolled copies included (2026-10-31)
TARGET_BASES = ("liquidation_debt_covered_value_usd", "net_liquidity_flow_usd",
                "avg_ltv", "borrow_capacity_utilization")

MAIN_COLS = (
    "borrow_amount_value_usd", "borrow_tx_count", "unique_borrowers",
      "repay_amount_value_usd", "repay_tx_count", "net_borrow_demand_usd",
        "borrow_repay_ratio", "borrow_growth",
        "supply_withdrawal_ratio", "liquidity_growth_rate", "withdrawal_amount_value_usd",
        "protocol_turnover_usd", "flashloan_amount_value_usd",
        "collateral_disabled_count", "collateral_enabled_count",
)

MARKET_COLS = (
    "asset_symbol", 
    "rate_change", "variable_borrow_rate",  
    "reserve_stress_score", 
    "liquidity_volatility", "liquidity_momentum_24h",
)
# --- columns kept for liquidation prediction (names only; a subset of cfg.FEATURE_COLS) ---
LIQ_COLS = (                    # what liquidations looked like today (from LIQ)
    "liquidation_tx_count", "unique_liquidated_users", "unique_liquidators",
    "liquidated_collateral_value_usd",
    "liquidation_rate", "liquidation_severity_usd", 
)
USER_COLS = (                   # how close sampled borrowers are to the edge (from USER)
    "avg_total_collateral_base", "avg_total_debt_base", "avg_available_borrows_base", "avg_current_liquidation_threshold",
    "ltv_utilization", "distance_to_liquidation",
    "risk_buffer",
)
FLOW_COLS = (                   # leverage and credit flows (from MAIN, rolled up to daily)
    "leverage_indicator", "borrow_repay_ratio", "net_borrow_demand_usd",
)
SELECTED_COLS = (*LIQ_COLS, *USER_COLS, *FLOW_COLS)

# --- denominators/operands the formulas below need; carried, NOT kept as features ---
# (kept out of SELECTED_COLS so the two asserts still describe the feature set only;
#  the liq/user ones are the grain-matched borrow context DF_*_features_24h already carries)
MAIN_INPUTS = ("supply_tx_count", "supply_amount_value_usd", "unique_repayers",
               "flashloan_tx_count", "unique_flashloan_initiators")
LIQ_INPUTS = ("borrow_tx_count", "unique_borrowers", "borrow_amount_value_usd")
USER_INPUTS = ("borrow_amount_value_usd",)

assert set(SELECTED_COLS) <= set(cfg.FEATURE_COLS), \
    sorted(set(SELECTED_COLS) - set(cfg.FEATURE_COLS))
assert len(SELECTED_COLS) == len(set(SELECTED_COLS)), "duplicate name in SELECTED_COLS"
assert not set(TARGET_BASES) & set((*MAIN_COLS, *SELECTED_COLS)), "a target is listed as a feature"
print(f" {len(SELECTED_COLS)} of {len(cfg.FEATURE_COLS)} features selected")

 16 of 53 features selected


In [6]:
# 2h panel + trailing rollups; ratio columns rebuilt as ratio-of-sums inside the SAME window(s)
df_daily =  MAIN[["time_bucket", *MAIN_COLS, *MAIN_INPUTS]]
ROLLUP_WINDOW, ROLLUP_SUFFIX = mds.TRAILING_WINDOWS, "rolled_{window}"   # 12h / 3d / 7d

ROLLUP_COLS_DAILY_COUNT = ( "borrow_tx_count",
                "supply_tx_count", "flashloan_tx_count",
                "collateral_enabled_count", "collateral_disabled_count" )
df_daily = mds.trailing_sums(df_daily, ROLLUP_COLS_DAILY_COUNT,
                             window=ROLLUP_WINDOW, suffix=ROLLUP_SUFFIX)

ROLLUP_COLS_DAILY_AMT = ( "borrow_amount_value_usd", "withdrawal_amount_value_usd", "supply_amount_value_usd", )
df_daily = mds.trailing_sums(df_daily, ROLLUP_COLS_DAILY_AMT,
                            window=ROLLUP_WINDOW, suffix=ROLLUP_SUFFIX)

# ratio-of-sums over the SAME window(s) as the rollup above, never the loaded 2h ratio:
# averaging twelve 2h ratios weights a $10 bucket like a $10M one (cfg.RATIO_RECIPES / SHARE_RECIPES)
# flashloan_user_activity has no recipe in cfg yet, so it is added here
RATIO_RECIPES = {**cfg.RATIO_RECIPES, **cfg.SHARE_RECIPES,
                 "flashloan_user_activity": ("flashloan_tx_count", "unique_flashloan_initiators")}
ROLLUP_COLS_DAILY_RATIO = (
    "borrow_repay_ratio", "leverage_indicator", "repayment_discipline",
    "flashloan_user_activity", "collateral_usage_rate", "flashloan_usage_intensity",
)
ROLLUP_RATIOS = {col: RATIO_RECIPES[col] for col in ROLLUP_COLS_DAILY_RATIO}
df_daily = mds.ratio_of_sums(df_daily, ROLLUP_RATIOS, window=None)          # per 2h bucket
df_daily = mds.ratio_of_sums(df_daily, ROLLUP_RATIOS, window=ROLLUP_WINDOW, suffix=ROLLUP_SUFFIX)

# drop raw bases + every rolled variant actually produced above, derived from the same naming
# rule trailing_sums used — never hand-written, so a window-set change can't leave a stale column
ROLLED_DROP_BASES = ("collateral_disabled_count", "supply_amount_value_usd", "withdrawal_amount_value_usd")
DROP_COLS = ("borrow_tx_count", "repay_tx_count", "supply_tx_count", "flashloan_tx_count", "borrow_tx_count_rolled_3d",
             "borrow_amount_value_usd_rolled_3d", "collateral_enabled_count_rolled_7d",
             "flashloan_user_activity_rolled_7d",
             "repayment_discipline_rolled_7d", "flashloan_usage_intensity_rolled_7d",
             *ROLLED_DROP_BASES,
             *[mds.trailing_name(base, w, ROLLUP_SUFFIX)
               for base in ROLLED_DROP_BASES for w, _ in mds.as_windows(ROLLUP_WINDOW)])
df_daily = df_daily.drop(columns=list(DROP_COLS))

df_daily


 trailing ['12h', '3d', '7d'] sums added for 5 column(s) (15 columns) · grain 0 days 02:00:00, a full window covers {'12h': 6, '3d': 36, '7d': 84} row(s)
 trailing ['12h', '3d', '7d'] sums added for 3 column(s) (9 columns) · grain 0 days 02:00:00, a full window covers {'12h': 6, '3d': 36, '7d': 84} row(s)
 6 ratio(s) rebuilt as ratio-of-sums per row
 6 ratio(s) rebuilt as ratio-of-sums over trailing ['12h', '3d', '7d'] (18 columns)


,time_bucket,borrow_amount_value_usd,unique_borrowers,repay_amount_value_usd,net_borrow_demand_usd,borrow_repay_ratio,borrow_growth,supply_withdrawal_ratio,liquidity_growth_rate,protocol_turnover_usd,...,leverage_indicator_rolled_7d,repayment_discipline_rolled_12h,repayment_discipline_rolled_3d,flashloan_user_activity_rolled_12h,flashloan_user_activity_rolled_3d,collateral_usage_rate_rolled_12h,collateral_usage_rate_rolled_3d,collateral_usage_rate_rolled_7d,flashloan_usage_intensity_rolled_12h,flashloan_usage_intensity_rolled_3d
0,2024-04-01 02:00:00.000 UTC,3.360310e+06,15.0,1.033550e+06,2.326760e+06,3.251231,NaN,0.917490,NaN,1.003003e+07,...,1.246027,0.400000,0.400000,1.000000,1.000000,0.514286,0.514286,0.514286,0.078947,0.078947
1,2024-04-01 04:00:00.000 UTC,5.841404e+06,19.0,1.544053e+06,4.297351e+06,3.783162,0.738353,27.477231,203.871678,6.030361e+07,...,0.171173,0.647059,0.647059,1.000000,1.000000,0.525424,0.525424,0.525424,0.101266,0.101266
2,2024-04-01 06:00:00.000 UTC,7.497619e+06,19.0,6.512004e+05,6.846418e+06,11.513535,0.283530,0.711785,-2.489268,4.433457e+08,...,0.071146,0.603774,0.603774,1.000000,1.000000,0.536585,0.536585,0.536585,0.085938,0.085938
3,2024-04-01 08:00:00.000 UTC,1.563728e+06,20.0,2.199827e+07,-2.043455e+07,0.071084,-0.791437,0.100023,0.418258,7.566389e+07,...,0.076269,0.575342,0.575342,1.000000,1.000000,0.542857,0.542857,0.542857,0.068783,0.068783
4,2024-04-01 10:00:00.000 UTC,7.696520e+06,20.0,5.185674e+06,2.510846e+06,1.484189,3.921904,0.850659,0.821135,1.073660e+08,...,0.091767,0.494624,0.494624,1.000000,1.000000,0.542683,0.542683,0.542683,0.053498,0.053498
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8731,2026-03-30 16:00:00.000 UTC,5.067923e+07,86.0,2.762900e+07,2.305023e+07,1.834277,2.502656,1.308038,5.684290,2.532291e+08,...,0.314736,0.706856,0.857219,1.869565,2.179245,0.511111,0.536015,0.539280,0.057796,0.060384
8732,2026-03-30 18:00:00.000 UTC,1.125271e+07,83.0,1.729953e+07,-6.046816e+06,0.650464,-0.777962,0.738458,-1.715809,1.396291e+08,...,0.313904,0.710526,0.845011,1.937500,2.149038,0.509434,0.535041,0.540204,0.056535,0.058294
8733,2026-03-30 20:00:00.000 UTC,9.916947e+06,74.0,9.870147e+06,4.680076e+04,1.004742,-0.118706,1.249990,1.282235,6.223628e+07,...,0.315433,0.779221,0.850802,1.960784,2.203980,0.508197,0.533208,0.539980,0.059032,0.058730
8734,2026-03-30 22:00:00.000 UTC,4.707848e+07,61.0,3.544758e+07,1.163091e+07,1.328116,3.747276,0.559306,-19.827946,3.967277e+08,...,0.318535,0.786177,0.851832,1.961538,2.110553,0.499121,0.531518,0.539059,0.061298,0.056082


In [7]:
# ORACLE_FILE = "query_result_data/batch_2025-04_2026-03/oracle_price_usd_eth_weth_6h_8403530_2025-04-01_2026-03-31.csv"

# PRICE_TOKENS = ["WBTC", "WETH"]

# px = pd.read_csv(
#     ORACLE_FILE,
#     parse_dates=["time_bucket"],
#     usecols=["time_bucket", "symbol", "avg_price_usd"],
# )
# px = px[px["symbol"].isin(PRICE_TOKENS)].copy()


In [8]:
selected_tokens = ["WBTC", "WETH", "USDT", "cbBTC", "wstETH"]

df_market_long = MARKET.loc[
    MARKET["asset_symbol"].isin(selected_tokens),
    ["time_bucket", *MARKET_COLS]
].copy()

# wide: one row per 2h bucket, one column per (feature, token) -- e.g. rate_change_WBTC
df_market = mds.pivot_wide_by_group(
    df_market_long, group_col="asset_symbol",
    value_cols=[c for c in MARKET_COLS if c != "asset_symbol"],
    groups=selected_tokens,
)

# a missing/NaN bucket means no market update, not a real gap -> carry the last value forward
df_market = mds.capped_ffill(df_market, limit=12)
df_market

 pivoted 5 column(s) x 5 group(s) -> 25 wide column(s), 59535 missing cell(s)
 forward-filled up to 12 bucket(s) (assumed unchanged): 47740 of 59535 NaN cell(s) filled across 25 column(s), 11795 left NaN beyond the cap


,time_bucket,rate_change_WBTC,rate_change_WETH,rate_change_USDT,rate_change_cbBTC,rate_change_wstETH,variable_borrow_rate_WBTC,variable_borrow_rate_WETH,variable_borrow_rate_USDT,variable_borrow_rate_cbBTC,...,liquidity_volatility_WBTC,liquidity_volatility_WETH,liquidity_volatility_USDT,liquidity_volatility_cbBTC,liquidity_volatility_wstETH,liquidity_momentum_24h_WBTC,liquidity_momentum_24h_WETH,liquidity_momentum_24h_USDT,liquidity_momentum_24h_cbBTC,liquidity_momentum_24h_wstETH
0,2024-04-01 02:00:00.000 UTC,NaN,-1.301234,0.543127,NaN,NaN,NaN,0.025638,0.056117,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-04-01 04:00:00.000 UTC,-5.313058,-1.301234,0.546496,NaN,NaN,0.008889,0.025615,0.056482,NaN,...,NaN,0.000024,0.000392,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-04-01 06:00:00.000 UTC,-5.313058,-1.297019,0.565208,NaN,NaN,0.008889,0.025679,0.059069,NaN,...,NaN,0.000046,0.002505,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-04-01 08:00:00.000 UTC,-5.313058,-3.755821,0.571877,NaN,NaN,0.008898,0.025489,0.059522,NaN,...,1.106666e-06,0.000115,0.002725,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-04-01 10:00:00.000 UTC,-5.313058,-3.745710,0.571071,NaN,NaN,0.008898,0.025554,0.059578,NaN,...,1.106666e-06,0.000114,0.002303,NaN,NaN,NaN,0.993417,1.127147,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8731,2026-03-30 16:00:00.000 UTC,-15.600286,-0.488300,0.901274,-6.324418,NaN,0.003326,0.022371,0.030962,0.003052,...,8.060163e-07,0.000068,0.000028,3.341999e-09,9.429282e-09,1.043473,1.005729,1.009485,1.000193,0.999068
8732,2026-03-30 18:00:00.000 UTC,-15.600286,-0.501219,0.901419,-6.325660,NaN,0.003327,0.022336,0.030939,0.003051,...,7.161180e-07,0.000065,0.000028,6.580596e-09,NaN,1.044788,1.005460,1.000820,0.999571,NaN
8733,2026-03-30 20:00:00.000 UTC,-15.702028,-0.504770,0.277265,-6.325660,NaN,0.003305,0.022320,0.030883,0.003051,...,6.649694e-07,0.000057,0.000044,6.580596e-09,NaN,1.002833,1.003276,0.996778,0.999571,NaN
8734,2026-03-30 22:00:00.000 UTC,-15.702028,-14.789524,0.901255,-6.325625,NaN,0.003304,0.022344,0.030902,0.003051,...,8.187419e-07,0.000032,0.000046,5.321275e-09,NaN,0.983951,1.005816,0.999490,0.999546,NaN


In [9]:
# target profile (info only): null %, zero %, spread per target
temp_market = adv.statistical_validation(df_market, columns=[c for c in df_market.columns
                                                            if c != "time_bucket"], save=False)
display(temp_market[["column", "null_pct", "zero_pct", "mean", "cv",]])

,column,null_pct,zero_pct,mean,cv
0,rate_change_WBTC,0.1030,0.0,-11.583981,1.218730
1,rate_change_WETH,0.0000,0.0,-1.196654,1.537788
2,rate_change_USDT,0.0000,0.0,0.433509,2.285525
3,rate_change_cbBTC,26.8086,0.0,-13.122267,2.461875
4,rate_change_wstETH,4.3842,0.0,-21.380874,0.995532
5,variable_borrow_rate_WBTC,0.0114,0.0,0.005574,0.569688
6,variable_borrow_rate_WETH,0.0000,0.0,0.025850,0.279566
7,variable_borrow_rate_USDT,0.0000,0.0,0.062638,0.460845
8,variable_borrow_rate_cbBTC,25.3434,0.0,0.002671,0.468525
9,variable_borrow_rate_wstETH,0.4006,0.0,0.003541,0.478698


In [10]:
# 2h grain: df_liq_2h placed on the panel's 2h grid, so df_liq has one row per df_daily row.
# df_liq_2h holds only buckets WITH a liquidation, so a missing bucket is a real 0 (no event).
# Borrow context (the denominators) comes from the 2h panel on the same grid, not from LIQ (daily).
LIQ_2H_COLS = ( "liquidation_tx_count", "unique_liquidated_users", "unique_liquidators",
                "liquidated_collateral_value_usd", "liquidation_debt_covered_value_usd", )
df_liq = mds.align_to_grid(run.load(SOURCES["liq_2h"])[["time_bucket", *LIQ_2H_COLS]],
                           MAIN[["time_bucket", *LIQ_INPUTS]], fill=0)

ROLLUP_COLS_LIQ = ( "liquidation_tx_count", "liquidated_collateral_value_usd", )
df_liq = mds.trailing_sums(df_liq, ROLLUP_COLS_LIQ,
                             window=ROLLUP_WINDOW, suffix=ROLLUP_SUFFIX)

LIQ_RATIO_RECIPES = {
    "avg_liquidation_debt_usd": ("liquidation_debt_covered_value_usd", "liquidation_tx_count"),
    "liquidation_rate":         ("liquidation_tx_count", "borrow_tx_count"),
    "liquidation_severity_usd": ("liquidated_collateral_value_usd", "liquidation_tx_count"),
    "liquidation_user_ratio":   ("unique_liquidated_users", "unique_borrowers"),
}
df_liq = mds.ratio_of_sums(df_liq, LIQ_RATIO_RECIPES, window=None)          # per 2h bucket

# market_stress_index_usd multiplies two flows, so it has no (num, den) recipe; zero borrow -> NaN
df_liq["market_stress_index_usd"] = (df_liq["liquidation_tx_count"] * df_liq["liquidation_debt_covered_value_usd"]) \
    / df_liq["borrow_amount_value_usd"].where(df_liq["borrow_amount_value_usd"] != 0)
DROP_LIQ_COLS = ("borrow_tx_count", "unique_borrowers", "borrow_amount_value_usd", "liquidation_tx_count",
                 "unique_liquidated_users", "unique_liquidators",
                  "liquidated_collateral_value_usd", "liquidation_debt_covered_value_usd",
                 "avg_liquidation_debt_usd", "liquidation_user_ratio", "liquidated_collateral_value_usd_rolled_12h",
                 "market_stress_index_usd",  "liquidated_collateral_value_usd_rolled_7d",
                # "liquidated_collateral_value_usd", "liquidation_debt_covered_value_usd",
                   )
df_liq = df_liq.drop(columns=list(DROP_LIQ_COLS))
df_liq

 aligned 2278 rows onto a 8736-row grid (6458 bucket(s) filled)
 trailing ['12h', '3d', '7d'] sums added for 2 column(s) (6 columns) · grain 0 days 02:00:00, a full window covers {'12h': 6, '3d': 36, '7d': 84} row(s)
 4 ratio(s) rebuilt as ratio-of-sums per row


,time_bucket,liquidation_tx_count_rolled_12h,liquidation_tx_count_rolled_3d,liquidation_tx_count_rolled_7d,liquidated_collateral_value_usd_rolled_3d,liquidation_rate,liquidation_severity_usd
0,2024-04-01 02:00:00.000 UTC,0.0,0.0,0.0,0.000000,0.000000,NaN
1,2024-04-01 04:00:00.000 UTC,0.0,0.0,0.0,0.000000,0.000000,NaN
2,2024-04-01 06:00:00.000 UTC,2.0,2.0,2.0,4212.421199,0.083333,2106.210599
3,2024-04-01 08:00:00.000 UTC,2.0,2.0,2.0,4212.421199,0.000000,NaN
4,2024-04-01 10:00:00.000 UTC,2.0,2.0,2.0,4212.421199,0.000000,NaN
...,...,...,...,...,...,...,...
8731,2026-03-30 16:00:00.000 UTC,0.0,48.0,144.0,113426.842764,0.000000,NaN
8732,2026-03-30 18:00:00.000 UTC,4.0,42.0,146.0,163634.979258,0.035398,14391.294494
8733,2026-03-30 20:00:00.000 UTC,6.0,40.0,148.0,144500.518868,0.019231,589.222341
8734,2026-03-30 22:00:00.000 UTC,6.0,40.0,148.0,144500.518868,0.000000,NaN


In [11]:
# PARKED 2026-10-29: superseded by the rollup inside cell 90c8b842, which drops these inputs (would KeyError)
# # trailing rollups of the 2h df_liq over ROLLUP_WINDOW (24h = the previous 12 buckets, closed at t);
# # ratios rebuilt as ratio-of-sums inside the SAME window, as for df_daily
# ROLLUP_COLS_LIQ_COUNT = ( "liquidation_tx_count", "unique_liquidated_users", "unique_liquidators", )
# df_liq = mds.trailing_sums(df_liq, ROLLUP_COLS_LIQ_COUNT,
#                            window=ROLLUP_WINDOW, suffix=ROLLUP_SUFFIX)
#
# ROLLUP_COLS_LIQ_AMT = ( "liquidated_collateral_value_usd", "liquidation_debt_covered_value_usd",
#                         "borrow_amount_value_usd", )        # borrow: operand of the stress index only
# df_liq = mds.trailing_sums(df_liq, ROLLUP_COLS_LIQ_AMT,
#                            window=ROLLUP_WINDOW, suffix=ROLLUP_SUFFIX)
#
# ROLLUP_COLS_LIQ_RATIO = ( "liquidation_rate", "liquidation_user_ratio", )
# ROLLUP_RATIOS_LIQ = {col: LIQ_RATIO_RECIPES[col] for col in ROLLUP_COLS_LIQ_RATIO}
# df_liq = mds.ratio_of_sums(df_liq, ROLLUP_RATIOS_LIQ, window=ROLLUP_WINDOW, suffix=ROLLUP_SUFFIX)
#
# # stress index from the rolled sums above, same formula as the per-bucket one
# TX, DEBT, BORROW = (f"{col}_{ROLLUP_SUFFIX}" for col in
#                     ("liquidation_tx_count", "liquidation_debt_covered_value_usd", "borrow_amount_value_usd"))
# df_liq[f"market_stress_index_usd_{ROLLUP_SUFFIX}"] = df_liq[TX] * df_liq[DEBT] / df_liq[BORROW].where(df_liq[BORROW] != 0)
# df_liq

In [12]:
# 2h grain: df_user_account_2h placed on the panel's 2h grid, so df_user has one row per df_daily row.
# It holds only SAMPLED buckets, so a missing bucket is unobserved: avg_* stay NaN, the weight is 0.
# borrow_amount_value_usd (capital_efficiency's numerator) comes from the 2h panel on the same grid.
# ltv_utilization must come before distance_to_liquidation (1 - it).
USER_WEIGHT = "sampled_user_count"      # sampled users behind each bucket's avg_*; weights the rollup
USER_2H_COLS = ( "avg_total_collateral_base", "avg_total_debt_base", "avg_available_borrows_base",
                 "avg_current_liquidation_threshold", "avg_ltv", )
df_user = mds.align_to_grid(run.load(SOURCES["user_2h"])[["time_bucket", *USER_2H_COLS, USER_WEIGHT]],
                            MAIN[["time_bucket", *USER_INPUTS]], fill={USER_WEIGHT: 0})

USER_RATIO_RECIPES = {
    "ltv_utilization":             ("avg_ltv", "avg_current_liquidation_threshold"),
    "debt_expansion_ratio":        ("avg_total_debt_base", "avg_total_collateral_base"),
    "capital_efficiency":          ("borrow_amount_value_usd", "avg_total_collateral_base"),
}
df_user = mds.ratio_of_sums(df_user, USER_RATIO_RECIPES, window=None)       # per 2h bucket

# differences, not ratios, so no recipe
df_user["distance_to_liquidation"]     = 1 - df_user["ltv_utilization"]
df_user["risk_buffer"]                 = df_user["avg_current_liquidation_threshold"] - df_user["avg_ltv"]
df_user["overcollateral_margin"]       = df_user["avg_total_collateral_base"] - df_user["avg_total_debt_base"]

# trailing rollups over USER_ROLLUP_WINDOW. avg_* do not sum across buckets, so
# they are weighted by sampled users (the mean of the users, not of the buckets).
# Narrower than the panel's ROLLUP_WINDOW: 12h is dropped here on purpose, since df_user holds
# only SAMPLED buckets (~50%) and a 12h window leaves ~364 of 4,368 rows NaN (A8). 3d/7d/15d are
# wide enough that an unsampled bucket inside them barely moves the weighted mean.
# Decided 2026-10-22: an unsampled bucket has weight 0 and adds nothing; a window with no sampled
# bucket at all stays NaN. Unobserved, not imputed.
# avg_ltv is a TARGET (2026-10-31): no trailing mean of it here; the targets section rolls it forward.
USER_ROLLUP_WINDOW = ("3d", "7d")

# ltv_utilization: both operands are user averages -> weighted by sampled users
ROLLUP_COLS_USER_RATIO_W = ( "ltv_utilization", )
ROLLUP_RATIOS_USER_W = {col: USER_RATIO_RECIPES[col] for col in ROLLUP_COLS_USER_RATIO_W}
df_user = mds.ratio_of_sums(df_user, ROLLUP_RATIOS_USER_W, window=USER_ROLLUP_WINDOW,
                            suffix=ROLLUP_SUFFIX, weight=USER_WEIGHT)

# capital_efficiency: a protocol flow over a user average -> unweighted
ROLLUP_COLS_USER_RATIO = ( "capital_efficiency", )
ROLLUP_RATIOS_USER = {col: USER_RATIO_RECIPES[col] for col in ROLLUP_COLS_USER_RATIO}
df_user = mds.ratio_of_sums(df_user, ROLLUP_RATIOS_USER, window=USER_ROLLUP_WINDOW, suffix=ROLLUP_SUFFIX)

DROP_USER_COLS = ("distance_to_liquidation","avg_available_borrows_base", "avg_current_liquidation_threshold", "ltv_utilization",
                   "debt_expansion_ratio", "avg_total_collateral_base",
                   "avg_ltv")             # avg_ltv: a target, kept above only as an operand
df_user = df_user.drop(columns=list(DROP_USER_COLS))
df_user

 aligned 3803 rows onto a 8736-row grid (4933 bucket(s) filled)
 3 ratio(s) rebuilt as ratio-of-sums per row
 1 ratio(s) rebuilt as ratio-of-sums over trailing ['3d', '7d'] (2 columns), weighted by sampled_user_count
 1 ratio(s) rebuilt as ratio-of-sums over trailing ['3d', '7d'] (2 columns)


,time_bucket,borrow_amount_value_usd,avg_total_debt_base,sampled_user_count,capital_efficiency,risk_buffer,overcollateral_margin,ltv_utilization_rolled_3d,ltv_utilization_rolled_7d,capital_efficiency_rolled_3d,capital_efficiency_rolled_7d
0,2024-04-01 02:00:00.000 UTC,3.360310e+06,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-04-01 04:00:00.000 UTC,5.841404e+06,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-04-01 06:00:00.000 UTC,7.497619e+06,3.971074e+06,1.0,1.706640e+00,0.02,422129.362910,0.978947,0.978947,1.706640,1.706640
3,2024-04-01 08:00:00.000 UTC,1.563728e+06,NaN,0.0,NaN,NaN,NaN,0.978947,0.978947,1.706640,1.706640
4,2024-04-01 10:00:00.000 UTC,7.696520e+06,NaN,0.0,NaN,NaN,NaN,0.978947,0.978947,1.706640,1.706640
...,...,...,...,...,...,...,...,...,...,...,...
8731,2026-03-30 16:00:00.000 UTC,5.067923e+07,NaN,0.0,NaN,NaN,NaN,0.968473,0.966593,28.648551,5.864980
8732,2026-03-30 18:00:00.000 UTC,1.125271e+07,3.864152e+00,1.0,1.484260e+06,0.02,3.717213,0.969695,0.966808,30.110901,5.941666
8733,2026-03-30 20:00:00.000 UTC,9.916947e+06,4.989394e+02,1.0,1.048560e+04,0.03,446.828418,0.969218,0.966734,32.573826,6.009210
8734,2026-03-30 22:00:00.000 UTC,4.707848e+07,NaN,0.0,NaN,NaN,NaN,0.969218,0.966734,32.573826,6.009210


In [13]:
# --- export the four 2h model-stage frames, one file each -----------------------------
# run.save() is transform.save_batch() pinned to THIS run's span: it writes
# model_data/batch_<start>_<end>/<table>_<first_date>_<last_date>.csv, deletes an older
# file of the same table whose dates changed, and MERGES _manifest.json (rows, cols,
# column names, dtypes, saved_utc per table) rather than replacing it.
# Dates in the name, unlike the fixed-name dataset at the end: these are intermediates,
# and publish_bundle's model_data allowlist takes the five fixed names only, so they
# stay local to this machine. Pass stage="inputs" to nest them one level deeper.
MODEL_FRAMES = {
    "DF_model_panel_2h":  df_daily,    # 2h protocol panel + trailing rollups + ratio-of-sums
    # "DF_model_market_2h": df_market,   # per-asset rate panel, selected tokens (several rows per bucket)
    # "DF_model_liq_2h":    df_liq,      # liquidations on the panel's 2h grid + trailing rollups
    # "DF_model_user_2h":   df_user,     # sampled user state on the 2h grid + weighted rollups
}
export_report = run.save(MODEL_FRAMES, out_dir=cfg.MODEL_DATA_DIR)
assert export_report["matches_batch"].all(), \
    export_report.loc[~export_report["matches_batch"], ["table", "span"]].to_string()
display(export_report)

wrote 1 tables -> model_data/batch_2024-04_2026-03/


,table,file,rows,cols,first_date,last_date,span,matches_batch,superseded
0,DF_model_panel_2h,DF_model_panel_2h_2024-04-01_2026-03-31.csv,8736,46,2024-04-01,2026-03-31,2024-04 → 2026-03,True,


## Scaled datasets — `DF_model_scaled_fixed_*` and `DF_model_scaled_causal_*` (leak-free)

Nothing is fitted on a train window, so both scales are the same for **any split proportion**.
Each frame's columns are first sorted into categories, then each column is shaped
(amounts `log_100(1 + sqrt(x))`, signed flows `asinh(x / S)`, counts `ln(1 + x)` / rolled
`log_10(1 + x)`, …) and mapped two ways, built side by side to compare:

- **fixed** — formulas + hand-set constants only (`SCALE_CONSTANTS`). The constants are
  **reference points, not limits**: for non-negative columns 100 = "at the reference", values
  above it keep rising (no cap) and only negatives are floored at 0 — shaped value ÷ shaped
  reference ceiling, `50·log2(1 + r)` for ratios (1 → 50), `×100` for shares,
  `100·asinh(x / c)/asinh(1)` for non-negative rates, a log range for tiny values. Signed
  columns stay bounded 0–100 with 50 = center (`tanh` for signed rates and momentum, a
  symmetric ceiling for signed flows). No statistic of the data, so a row's score never
  depends on any other row.
- **causal** — each row min-max scaled against **its own past only**: the min and max of the
  shaped column over the trailing `CAUSAL_WINDOW`, so always 0–100. Leak-free by construction; the first
  `CAUSAL_MIN_ROWS` rows of each column are NaN (warm-up).

Binaries are left as they are. The raw frames above are not changed.
Plumbing: `src/model_scaling.py`.

In [14]:
import model_scaling as msc

# --- scaling config: each frame's columns sorted into ONE category (rules: msc.CATEGORY_RULES) ---
# A base name also claims its rolled variants, one per window in the active rollup window set
# (see ROLLED_SUFFIXES in the next cell), even when the base column itself was dropped above.
PANEL_TYPES = {
    "amount":        ("borrow_amount_value_usd", "repay_amount_value_usd", "protocol_turnover_usd",
                      "flashloan_amount_value_usd"),
    "signed_amount": ("net_borrow_demand_usd",),
    "count":         ("collateral_enabled_count", "borrow_tx_count",
                      "supply_tx_count", "flashloan_tx_count"),
    "unique_count":  ("unique_borrowers", "unique_repayers", "unique_flashloan_initiators"),
    "ratio":         ("borrow_repay_ratio", "supply_withdrawal_ratio", "leverage_indicator",
                      "repayment_discipline", "flashloan_user_activity"),
    "share":         ("collateral_usage_rate", "flashloan_usage_intensity"),
    "signed_rate":   ("borrow_growth", "liquidity_growth_rate"),
}
MARKET_TYPES = {
    "signed_rate":   ("rate_change",),
    "positive_rate": ("variable_borrow_rate",),
    "ratio":         ("reserve_stress_score",),
    "momentum":      ("liquidity_momentum_24h",),
    "tiny_positive": ("liquidity_volatility",),
}
LIQ_TYPES = {
    "count":         ("liquidation_tx_count",),
    "amount":        ("liquidated_collateral_value_usd", "liquidation_severity_usd"),
    "positive_rate": ("liquidation_rate",),
}
USER_TYPES = {
    "amount":        ("borrow_amount_value_usd", "avg_total_debt_base"),
    "signed_amount": ("overcollateral_margin",),
    "signed_rate":   ("ltv_utilization",),         # sits at 0.96-0.98: a centred tanh shows its moves
    "positive_rate": ("risk_buffer",),
    "ratio":         ("capital_efficiency",),
    "skip":          ("sampled_user_count",),      # the rollup weight, not a feature
}
HEAVY_RATIOS = ("flashloan_user_activity", "reserve_stress_score",
                "capital_efficiency")              # causal shape log10(1 + r): spikes would flatten a window

# --- FIXED constants: set by hand from domain knowledge, frozen; never recomputed from data ---
SCALE_CONSTANTS = {
    # REFERENCE raw value -> 100 (shaped the same way as the column); larger values score above 100
    # counts AND unique counts share "count" (ln(1+x)); only rolled ones use "count_rolled" (log10)
    "ceiling": {"amount": 1e11, "count": 1e4, "count_rolled": 1e5},
    # signed flows: asinh(x / S), mapped so -ceiling -> 0, 0 -> 50, +ceiling -> 100
    "asinh_scale": {"net_borrow_demand_usd": 1e6, "net_liquidity_flow_usd": 1e6,   # target only
                    "overcollateral_margin": 1.0},
    "asinh_ceiling": 1e10,
    # 50 + 50·tanh((x - center) / c): near-linear within +-c of center
    "tanh": {"borrow_growth": (0.0, 1.0), "liquidity_growth_rate": (0.0, 2.0),
             "rate_change": (0.0, 1.0), "borrow_supply_spread": (0.01, 0.01),
             "borrow_momentum_24h": (1.0, 0.01), "liquidity_momentum_24h": (1.0, 0.01),
             "ltv_utilization": (0.95, 0.02)},
    # 100·asinh(x / c) / asinh(1) for x >= 0: 0 -> 0, x = c -> 100, larger keeps rising
    "positive_tanh": {"variable_borrow_rate": 0.05, "liquidation_rate": 0.05,
                      "market_stress_index_usd": 0.01, "risk_buffer": 0.03},
    # tiny positives: log10 range 1e-9 -> 0, 1 -> 100, larger keeps rising
    "log_range": (-9, 0),
}

# --- CAUSAL: each row against its own trailing window only :::: USED FOR MIN MAX SCALING USING PREVIOUS DATA ---
CAUSAL_WINDOW = "30d"
CAUSAL_MIN_ROWS = 84                # 7 days of 2h rows observed before a score is given

# frame label -> (raw frame, its categories, group column scaled separately)
FRAMES = {
    "panel":  (df_daily,  PANEL_TYPES,  None),
    "market": (df_market, MARKET_TYPES, None),   # wide now: each token already its own column
    "liq":    (df_liq,    LIQ_TYPES,    None),
    "user":   (df_user,   USER_TYPES,   None),
}

In [15]:
# every rolled suffix that could appear anywhere below (panel/liq's ROLLUP_WINDOW union
# df_user's narrower USER_ROLLUP_WINDOW); _variants only keeps the ones that actually exist as
# columns in a given frame, so passing this superset to every frame is safe, market included
# (market has no rolled columns at all, so none of these suffixes match anything there)
ROLLED_SUFFIXES = tuple(dict.fromkeys((*mds.window_suffixes(ROLLUP_WINDOW, ROLLUP_SUFFIX),
                                       *mds.window_suffixes(USER_ROLLUP_WINDOW, ROLLUP_SUFFIX))))

# categorise: every column of each frame gets exactly one rule; raises if one is missed or doubled
COLMAPS = {name: msc.categorize_columns(frame, types, heavy=HEAVY_RATIOS, suffixes=ROLLED_SUFFIXES,
                                        groups=selected_tokens if name == "market" else ())
           for name, (frame, types, _) in FRAMES.items()}
for name, colmap in COLMAPS.items():
    print(f"== {name}: {len(colmap)} columns")
    display(msc.category_table(colmap))

== panel: 46 columns


,column,category,source,shape,base,fixed,rolled,group
0,borrow_amount_value_usd,amount,borrow_amount_value_usd,sqrt_log,100.0000,ceiling,False,None
1,borrow_amount_value_usd_rolled_12h,amount,borrow_amount_value_usd,sqrt_log,100.0000,ceiling,True,None
2,borrow_amount_value_usd_rolled_7d,amount,borrow_amount_value_usd,sqrt_log,100.0000,ceiling,True,None
3,flashloan_amount_value_usd,amount,flashloan_amount_value_usd,sqrt_log,100.0000,ceiling,False,None
4,protocol_turnover_usd,amount,protocol_turnover_usd,sqrt_log,100.0000,ceiling,False,None
5,repay_amount_value_usd,amount,repay_amount_value_usd,sqrt_log,100.0000,ceiling,False,None
6,borrow_tx_count_rolled_12h,count,borrow_tx_count,log1p,10.0000,ceiling,True,None
7,borrow_tx_count_rolled_7d,count,borrow_tx_count,log1p,10.0000,ceiling,True,None
8,collateral_enabled_count,count,collateral_enabled_count,log1p,2.7183,ceiling,False,None
9,collateral_enabled_count_rolled_12h,count,collateral_enabled_count,log1p,10.0000,ceiling,True,None


== market: 26 columns


,column,category,source,shape,base,fixed,rolled,group
0,liquidity_momentum_24h_USDT,momentum,liquidity_momentum_24h,identity,NaN,tanh,False,USDT
1,liquidity_momentum_24h_WBTC,momentum,liquidity_momentum_24h,identity,NaN,tanh,False,WBTC
2,liquidity_momentum_24h_WETH,momentum,liquidity_momentum_24h,identity,NaN,tanh,False,WETH
3,liquidity_momentum_24h_cbBTC,momentum,liquidity_momentum_24h,identity,NaN,tanh,False,cbBTC
4,liquidity_momentum_24h_wstETH,momentum,liquidity_momentum_24h,identity,NaN,tanh,False,wstETH
5,variable_borrow_rate_USDT,positive_rate,variable_borrow_rate,identity,NaN,positive_tanh,False,USDT
6,variable_borrow_rate_WBTC,positive_rate,variable_borrow_rate,identity,NaN,positive_tanh,False,WBTC
7,variable_borrow_rate_WETH,positive_rate,variable_borrow_rate,identity,NaN,positive_tanh,False,WETH
8,variable_borrow_rate_cbBTC,positive_rate,variable_borrow_rate,identity,NaN,positive_tanh,False,cbBTC
9,variable_borrow_rate_wstETH,positive_rate,variable_borrow_rate,identity,NaN,positive_tanh,False,wstETH


== liq: 7 columns


,column,category,source,shape,base,fixed,rolled,group
0,liquidated_collateral_value_usd_rolled_3d,amount,liquidated_collateral_value_usd,sqrt_log,100.0,ceiling,True,None
1,liquidation_severity_usd,amount,liquidation_severity_usd,sqrt_log,100.0,ceiling,False,None
2,liquidation_tx_count_rolled_12h,count,liquidation_tx_count,log1p,10.0,ceiling,True,None
3,liquidation_tx_count_rolled_3d,count,liquidation_tx_count,log1p,10.0,ceiling,True,None
4,liquidation_tx_count_rolled_7d,count,liquidation_tx_count,log1p,10.0,ceiling,True,None
5,liquidation_rate,positive_rate,liquidation_rate,identity,NaN,positive_tanh,False,None
6,time_bucket,skip,time_bucket,identity,NaN,none,False,None


== user: 11 columns


,column,category,source,shape,base,fixed,rolled,group
0,avg_total_debt_base,amount,avg_total_debt_base,sqrt_log,100.0,ceiling,False,None
1,borrow_amount_value_usd,amount,borrow_amount_value_usd,sqrt_log,100.0,ceiling,False,None
2,risk_buffer,positive_rate,risk_buffer,identity,NaN,positive_tanh,False,None
3,capital_efficiency,ratio,capital_efficiency,log1p,10.0,odds,False,None
4,capital_efficiency_rolled_3d,ratio,capital_efficiency,log1p,10.0,odds,True,None
5,capital_efficiency_rolled_7d,ratio,capital_efficiency,log1p,10.0,odds,True,None
6,overcollateral_margin,signed_amount,overcollateral_margin,asinh,NaN,symmetric_ceiling,False,None
7,ltv_utilization_rolled_3d,signed_rate,ltv_utilization,identity,NaN,tanh,True,None
8,ltv_utilization_rolled_7d,signed_rate,ltv_utilization,identity,NaN,tanh,True,None
9,sampled_user_count,skip,sampled_user_count,identity,NaN,none,False,None


In [16]:
# check each shape's domain before fitting: a negative in a sqrt/log column is a definition bug
for name, (frame, _, _) in FRAMES.items():
    check = msc.check_category_assumptions(frame, COLMAPS[name])
    print(f"== {name}: {int((~check['domain_ok']).sum())} column(s) outside their shape's domain")
    display(check)

== panel: 0 column(s) outside their shape's domain


,column,category,shape,min,max,neg_pct,zero_pct,nan_pct,domain_ok
0,borrow_amount_value_usd,amount,sqrt_log,6.735490e+04,1.159603e+09,0.00,0.00,0.00,True
1,borrow_amount_value_usd_rolled_12h,amount,sqrt_log,3.360310e+06,1.808996e+09,0.00,0.00,0.00,True
2,borrow_amount_value_usd_rolled_7d,amount,sqrt_log,3.360310e+06,1.058982e+10,0.00,0.00,0.00,True
3,repay_amount_value_usd,amount,sqrt_log,1.807678e+04,1.118390e+09,0.00,0.00,0.00,True
4,protocol_turnover_usd,amount,sqrt_log,1.277055e+06,3.830098e+10,0.00,0.00,0.00,True
5,flashloan_amount_value_usd,amount,sqrt_log,0.000000e+00,4.219854e+09,0.00,1.26,0.00,True
6,net_borrow_demand_usd,signed_amount,asinh,-5.791846e+08,4.868190e+08,45.94,0.00,0.00,True
7,collateral_enabled_count,count,log1p,6.000000e+00,1.260000e+03,0.00,0.00,0.00,True
8,collateral_enabled_count_rolled_12h,count,log1p,1.800000e+01,3.322000e+03,0.00,0.00,0.00,True
9,collateral_enabled_count_rolled_3d,count,log1p,1.800000e+01,1.019500e+04,0.00,0.00,0.00,True


== market: 0 column(s) outside their shape's domain


,column,category,shape,min,max,neg_pct,zero_pct,nan_pct,domain_ok
0,rate_change_WBTC,signed_rate,identity,-3.409113e+02,0.999966,89.73,0.0,0.10,True
1,rate_change_WETH,signed_rate,identity,-3.490482e+01,0.999994,80.46,0.0,0.00,True
2,rate_change_USDT,signed_rate,identity,-1.851323e+01,0.999865,15.83,0.0,0.00,True
3,rate_change_cbBTC,signed_rate,identity,-1.453126e+03,0.998249,71.26,0.0,26.81,True
4,rate_change_wstETH,signed_rate,identity,-3.275551e+02,0.913476,92.19,0.0,4.38,True
5,variable_borrow_rate_WBTC,positive_rate,identity,1.354926e-03,0.013000,0.00,0.0,0.01,True
6,variable_borrow_rate_WETH,positive_rate,identity,1.827328e-02,0.226435,0.00,0.0,0.00,True
7,variable_borrow_rate_USDT,positive_rate,identity,2.715639e-02,0.759164,0.00,0.0,0.00,True
8,variable_borrow_rate_cbBTC,positive_rate,identity,1.819030e-05,0.008456,0.00,0.0,25.34,True
9,variable_borrow_rate_wstETH,positive_rate,identity,4.608686e-04,0.008073,0.00,0.0,0.40,True


== liq: 0 column(s) outside their shape's domain


,column,category,shape,min,max,neg_pct,zero_pct,nan_pct,domain_ok
0,liquidation_tx_count_rolled_12h,count,log1p,0.0,4.538000e+03,0.0,28.07,0.00,True
1,liquidation_tx_count_rolled_3d,count,log1p,0.0,5.804000e+03,0.0,0.64,0.00,True
2,liquidation_tx_count_rolled_7d,count,log1p,0.0,1.054600e+04,0.0,0.02,0.00,True
3,liquidated_collateral_value_usd_rolled_3d,amount,sqrt_log,0.0,2.659708e+08,0.0,0.64,0.00,True
4,liquidation_severity_usd,amount,sqrt_log,0.0,2.528382e+06,0.0,0.06,73.92,True
5,liquidation_rate,positive_rate,identity,0.0,2.466667e+01,0.0,73.92,0.00,True


== user: 0 column(s) outside their shape's domain


,column,category,shape,min,max,neg_pct,zero_pct,nan_pct,domain_ok
0,borrow_amount_value_usd,amount,sqrt_log,67354.896435,1.159603e+09,0.00,0.00,0.00,True
1,avg_total_debt_base,amount,sqrt_log,0.000000,5.101165e+08,0.00,2.93,56.47,True
2,overcollateral_margin,signed_amount,asinh,-0.789634,3.509693e+08,0.03,0.85,56.47,True
3,ltv_utilization_rolled_3d,signed_rate,identity,0.852578,9.789474e-01,0.00,0.00,0.02,True
4,ltv_utilization_rolled_7d,signed_rate,identity,0.914680,9.789474e-01,0.00,0.00,0.02,True
5,risk_buffer,positive_rate,identity,0.000000,7.500000e-01,0.00,0.85,56.47,True
6,capital_efficiency,ratio,log1p,0.000431,4.285133e+15,0.00,0.00,57.31,True
7,capital_efficiency_rolled_3d,ratio,log1p,0.024921,8.664404e+02,0.00,0.00,0.02,True
8,capital_efficiency_rolled_7d,ratio,log1p,0.095873,3.622243e+01,0.00,0.00,0.02,True


In [17]:
# FIXED scaling: formula + hand-set constant per column, no statistic of the data -> same for any split
SCALED_FIXED = {name: msc.scale_fixed(frame, COLMAPS[name], SCALE_CONSTANTS)
                for name, (frame, _, _) in FRAMES.items()}
DF_model_scaled_fixed_panel  = SCALED_FIXED["panel"]
DF_model_scaled_fixed_market = SCALED_FIXED["market"]
DF_model_scaled_fixed_liq    = SCALED_FIXED["liq"]
DF_model_scaled_fixed_user   = SCALED_FIXED["user"]
DF_model_scaled_fixed_panel

 fixed scaling: 45 column(s), no statistic of the data used
 fixed scaling: 25 column(s), no statistic of the data used
 fixed scaling: 6 column(s), no statistic of the data used
 fixed scaling: 9 column(s), no statistic of the data used


,time_bucket,borrow_amount_value_usd,unique_borrowers,repay_amount_value_usd,net_borrow_demand_usd,borrow_repay_ratio,borrow_growth,supply_withdrawal_ratio,liquidity_growth_rate,protocol_turnover_usd,...,leverage_indicator_rolled_7d,repayment_discipline_rolled_12h,repayment_discipline_rolled_3d,flashloan_user_activity_rolled_12h,flashloan_user_activity_rolled_3d,collateral_usage_rate_rolled_12h,collateral_usage_rate_rolled_3d,collateral_usage_rate_rolled_7d,flashloan_usage_intensity_rolled_12h,flashloan_usage_intensity_rolled_3d
0,2024-04-01 02:00:00.000 UTC,59.335013,30.102673,54.683491,57.981514,104.394027,NaN,46.960947,NaN,63.650678,...,58.368748,24.271341,24.271341,50.000000,50.000000,51.428571,51.428571,51.428571,7.894737,7.894737
1,2024-04-01 04:00:00.000 UTC,61.517044,32.525397,56.266904,60.927547,112.898235,81.407448,241.586849,1.000000e+02,70.731393,...,11.397728,35.994604,35.994604,50.000000,50.000000,52.542373,52.542373,52.542373,10.126582,10.126582
2,2024-04-01 06:00:00.000 UTC,62.502171,32.525397,52.861717,63.238590,182.270876,63.808464,38.775091,7.661394e+00,78.607105,...,4.957783,34.073524,34.073524,50.000000,50.000000,53.658537,53.658537,53.658537,8.593750,8.593750
3,2024-04-01 08:00:00.000 UTC,56.316855,33.055123,66.750654,31.264316,4.953591,17.038893,6.876695,6.030663e+01,71.627150,...,5.301945,32.783275,32.783275,50.000000,50.000000,54.285714,54.285714,54.285714,6.878307,6.878307
4,2024-04-01 10:00:00.000 UTC,62.605507,33.055123,61.047135,58.336719,65.638746,99.960798,44.401963,6.944771e+01,73.008623,...,6.333265,28.989113,28.989113,50.000000,50.000000,54.268293,54.268293,54.268293,5.349794,5.349794
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8731,2026-03-30 16:00:00.000 UTC,70.045000,48.487455,67.650259,69.343151,75.149031,99.334237,60.333358,9.966126e+01,76.396055,...,19.738635,38.567059,44.657196,76.041608,83.434216,51.111111,53.601533,53.927964,5.779570,6.038426
8732,2026-03-30 18:00:00.000 UTC,64.104677,48.106460,65.802194,37.380974,36.143564,17.423231,39.890418,1.524118e+01,74.045886,...,19.692976,38.722015,44.181456,77.729443,82.745569,50.943396,53.504145,54.020385,5.653495,5.829421
8733,2026-03-30 20:00:00.000 UTC,63.605928,46.876023,63.587257,50.236198,50.170816,44.092411,58.495940,7.828300e+01,70.855925,...,19.776884,41.562277,44.407534,78.298970,83.993259,50.819672,53.320826,53.998026,5.903188,5.872995
8734,2026-03-30 22:00:00.000 UTC,69.754065,44.809306,68.633914,65.896704,60.958139,99.944420,32.045182,2.448118e-07,78.168491,...,19.946807,41.843757,44.447657,78.317341,81.858549,49.912127,53.151809,53.905919,6.129808,5.608225


In [18]:
# CAUSAL scaling: each row min-max scaled against its own trailing window (no row after t is read);
# market scaled per asset
SCALED_CAUSAL = {name: msc.scale_causal(frame, COLMAPS[name], SCALE_CONSTANTS, window=CAUSAL_WINDOW,
                                        min_rows=CAUSAL_MIN_ROWS, by=by)
                 for name, (frame, _, by) in FRAMES.items()}
DF_model_scaled_causal_panel  = SCALED_CAUSAL["panel"]
DF_model_scaled_causal_market = SCALED_CAUSAL["market"]
DF_model_scaled_causal_liq    = SCALED_CAUSAL["liq"]
DF_model_scaled_causal_user   = SCALED_CAUSAL["user"]
DF_model_scaled_causal_panel

 causal scaling over trailing 30d (min 84 rows): 45 column(s), 91 row(s) with warm-up NaN
 causal scaling over trailing 30d (min 84 rows): 25 column(s), 201 row(s) with warm-up NaN
 causal scaling over trailing 30d (min 84 rows): 6 column(s), 959 row(s) with warm-up NaN
 causal scaling over trailing 30d (min 84 rows): 9 column(s), 235 row(s) with warm-up NaN


,time_bucket,borrow_amount_value_usd,unique_borrowers,repay_amount_value_usd,net_borrow_demand_usd,borrow_repay_ratio,borrow_growth,supply_withdrawal_ratio,liquidity_growth_rate,protocol_turnover_usd,...,leverage_indicator_rolled_7d,repayment_discipline_rolled_12h,repayment_discipline_rolled_3d,flashloan_user_activity_rolled_12h,flashloan_user_activity_rolled_3d,collateral_usage_rate_rolled_12h,collateral_usage_rate_rolled_3d,collateral_usage_rate_rolled_7d,flashloan_usage_intensity_rolled_12h,flashloan_usage_intensity_rolled_3d
0,2024-04-01 02:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-04-01 04:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-04-01 06:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-04-01 08:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-04-01 10:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8731,2026-03-30 16:00:00.000 UTC,70.930433,62.786283,65.772833,87.042072,4.469135,14.261168,9.468498,94.109575,57.028705,...,24.858543,16.033450,59.957644,23.263754,23.140893,35.549559,73.600344,92.015170,53.299691,57.189978
8732,2026-03-30 18:00:00.000 UTC,45.131129,60.962114,58.750287,27.665611,1.488872,0.823340,5.072105,92.976809,43.567245,...,24.467189,16.711023,54.433033,25.412021,20.965410,34.608844,72.297341,93.701596,51.815139,49.452279
8733,2026-03-30 20:00:00.000 UTC,42.965021,55.070891,50.333647,51.546892,2.380771,3.523735,9.020446,93.435733,25.295572,...,25.186574,29.391919,57.053855,26.136912,24.906947,33.914874,69.844620,93.293601,54.755315,51.065450
8734,2026-03-30 22:00:00.000 UTC,69.666876,45.175636,69.510669,80.639572,3.194869,19.359288,3.689287,90.204303,67.180984,...,26.645962,30.676047,57.519848,26.160295,18.163256,28.824440,67.583262,91.612915,57.423810,41.263280


In [19]:
# QA per frame: each method vs raw, then fixed vs causal.
# fixed must be monotone (spearman 1); causal is relative to each row's own window, so it is not.
# bunched -> IQR under 5 points; saturated -> over 20% of rows at 0 or 100 (fix the constant)
for name, (frame, _, by) in FRAMES.items():
    print(f"== {name}: fixed")
    display(msc.scaling_report(frame, SCALED_FIXED[name], COLMAPS[name], by=by))
    print(f"== {name}: causal")
    display(msc.scaling_report(frame, SCALED_CAUSAL[name], COLMAPS[name], by=by))
    print(f"== {name}: fixed vs causal")
    display(msc.compare_scalings(SCALED_FIXED[name], SCALED_CAUSAL[name], COLMAPS[name], by=by))

== panel: fixed


,column,category,fixed,min,p25,p50,p75,max,iqr,pct_at_0,pct_at_100,pct_above_ref,pct_nan_added,monotone,spearman,bunched,pct_at_bound
0,borrow_amount_value_usd,amount,ceiling,43.925,62.433,65.658,68.954,82.403,6.521,0.000,0.000,0.000,0.0,True,1.0,False,0.000
1,borrow_amount_value_usd_rolled_12h,amount,ceiling,59.335,71.248,73.890,76.376,84.159,5.128,0.000,0.000,0.000,0.0,True,1.0,False,0.000
2,borrow_amount_value_usd_rolled_7d,amount,ceiling,59.335,82.723,84.871,86.506,91.135,3.784,0.000,0.000,0.000,0.0,True,1.0,True,0.000
3,repay_amount_value_usd,amount,ceiling,38.760,62.186,65.412,68.669,82.260,6.482,0.000,0.000,0.000,0.0,True,1.0,False,0.000
4,protocol_turnover_usd,amount,ceiling,55.518,73.054,76.616,79.593,96.211,6.538,0.000,0.000,0.000,0.0,True,1.0,False,0.000
5,flashloan_amount_value_usd,amount,ceiling,0.000,49.017,53.407,57.211,87.503,8.194,1.259,0.000,0.000,0.0,True,1.0,False,1.259
6,net_borrow_demand_usd,signed_amount,symmetric_ceiling,14.382,38.169,52.031,63.471,84.740,25.302,0.000,0.000,0.000,0.0,True,1.0,False,0.000
7,collateral_enabled_count,count,ceiling,21.127,39.776,45.322,50.214,77.517,10.438,0.000,0.000,0.000,0.0,True,1.0,False,0.000
8,collateral_enabled_count_rolled_12h,count,ceiling,25.575,47.712,52.149,55.865,70.431,8.153,0.000,0.000,0.000,0.0,True,1.0,False,0.000
9,collateral_enabled_count_rolled_3d,count,ceiling,25.575,63.568,68.263,71.550,80.169,7.982,0.000,0.000,0.000,0.0,True,1.0,False,0.000


== panel: causal


,column,category,fixed,min,p25,p50,p75,max,iqr,pct_at_0,pct_at_100,pct_above_ref,pct_nan_added,monotone,spearman,bunched,pct_at_bound
0,borrow_amount_value_usd,amount,ceiling,0.0,43.722,55.539,66.865,100.0,23.143,0.424,0.401,0.0,0.95,False,0.735,False,0.825
1,borrow_amount_value_usd_rolled_12h,amount,ceiling,0.0,39.612,55.415,70.897,100.0,31.285,0.469,0.950,0.0,0.95,False,0.580,False,1.419
2,borrow_amount_value_usd_rolled_7d,amount,ceiling,0.0,26.290,62.784,90.709,100.0,64.419,3.114,5.449,0.0,0.95,False,0.149,False,8.563
3,repay_amount_value_usd,amount,ceiling,0.0,45.934,58.179,69.363,100.0,23.429,0.321,0.469,0.0,0.95,False,0.634,False,0.790
4,protocol_turnover_usd,amount,ceiling,0.0,39.726,54.338,69.577,100.0,29.852,0.401,0.412,0.0,0.95,False,0.816,False,0.813
5,flashloan_amount_value_usd,amount,ceiling,0.0,58.425,69.752,77.834,100.0,19.409,1.259,0.229,0.0,0.95,False,0.533,False,1.488
6,net_borrow_demand_usd,signed_amount,symmetric_ceiling,0.0,27.927,51.046,72.519,100.0,44.591,0.446,0.412,0.0,0.95,False,0.991,False,0.858
7,collateral_enabled_count,count,ceiling,0.0,32.330,43.290,54.945,100.0,22.615,0.481,0.469,0.0,0.95,False,0.373,False,0.950
8,collateral_enabled_count_rolled_12h,count,ceiling,0.0,29.245,43.179,60.333,100.0,31.088,0.790,1.156,0.0,0.95,False,0.376,False,1.946
9,collateral_enabled_count_rolled_3d,count,ceiling,0.0,23.263,46.310,73.292,100.0,50.028,2.942,3.835,0.0,0.95,False,0.264,False,6.777


== panel: fixed vs causal


,column,category,spearman_fixed_vs_causal,median_abs_gap,iqr_fixed,iqr_causal
0,borrow_amount_value_usd,amount,0.735,11.621,6.521,23.143
1,borrow_amount_value_usd_rolled_12h,amount,0.580,19.216,5.128,31.285
2,borrow_amount_value_usd_rolled_7d,amount,0.149,22.382,3.784,64.419
3,repay_amount_value_usd,amount,0.634,10.534,6.482,23.429
4,protocol_turnover_usd,amount,0.816,21.888,6.538,29.852
5,flashloan_amount_value_usd,amount,0.533,19.066,8.194,19.409
6,net_borrow_demand_usd,signed_amount,0.991,9.372,25.302,44.591
7,collateral_enabled_count,count,0.373,11.229,10.438,22.615
8,collateral_enabled_count_rolled_12h,count,0.376,16.655,8.153,31.088
9,collateral_enabled_count_rolled_3d,count,0.264,28.043,7.982,50.028


... (14 outputs left out)

== user: causal


,column,category,fixed,min,p25,p50,p75,max,iqr,pct_at_0,pct_at_100,pct_above_ref,pct_nan_added,monotone,spearman,bunched,pct_at_bound
0,borrow_amount_value_usd,amount,ceiling,0.0,43.722,55.539,66.865,100.0,23.143,0.424,0.401,0.0,0.950,False,0.735,False,0.825
1,avg_total_debt_base,amount,ceiling,0.0,50.267,68.341,88.711,100.0,38.444,2.873,0.446,0.0,1.763,False,0.988,False,3.319
2,overcollateral_margin,signed_amount,symmetric_ceiling,0.0,52.728,70.233,88.727,100.0,35.999,0.801,0.561,0.0,1.763,False,0.984,False,1.362
3,ltv_utilization_rolled_3d,signed_rate,tanh,0.0,36.929,57.133,75.883,100.0,38.954,1.465,1.316,0.0,0.950,False,0.831,False,2.781
4,ltv_utilization_rolled_7d,signed_rate,tanh,0.0,24.865,51.044,77.378,100.0,52.512,2.919,3.228,0.0,0.950,False,0.755,False,6.147
5,risk_buffer,positive_rate,positive_tanh,0.0,25.000,35.714,49.288,100.0,24.288,0.824,0.836,0.0,1.763,False,0.688,False,1.660
6,capital_efficiency,ratio,odds,0.0,2.992,16.806,34.600,100.0,31.608,0.252,0.229,0.0,1.889,False,0.962,False,0.481
7,capital_efficiency_rolled_3d,ratio,odds,0.0,8.576,22.786,47.890,100.0,39.314,1.706,1.751,0.0,0.950,False,0.654,False,3.457
8,capital_efficiency_rolled_7d,ratio,odds,0.0,14.414,34.273,65.175,100.0,50.761,2.804,2.793,0.0,0.950,False,0.575,False,5.597


== user: fixed vs causal


,column,category,spearman_fixed_vs_causal,median_abs_gap,iqr_fixed,iqr_causal
0,borrow_amount_value_usd,amount,0.735,11.621,6.521,23.143
1,avg_total_debt_base,amount,0.988,16.630,27.705,38.444
2,overcollateral_margin,signed_amount,0.984,10.024,14.431,35.999
3,ltv_utilization_rolled_3d,signed_rate,0.831,25.144,10.320,38.954
4,ltv_utilization_rolled_7d,signed_rate,0.755,30.889,8.913,52.512
5,risk_buffer,positive_rate,0.688,50.266,29.071,24.288
6,capital_efficiency,ratio,0.962,208.946,403.735,31.608
7,capital_efficiency_rolled_3d,ratio,0.654,22.164,68.453,39.314
8,capital_efficiency_rolled_7d,ratio,0.575,20.851,58.996,50.761


In [20]:
# --- export both scaled versions + the config that reproduces them (same folder as the raw exports) ---
# scaling_config.json holds only hand-set constants and the causal window — nothing fitted.
SCALED_FRAMES_FIXED = {
    **{f"DF_model_scaled_fixed_{name}_2h": df for name, df in SCALED_FIXED.items()},
}
scaled_export_fixed = run.save(SCALED_FRAMES_FIXED, out_dir=cfg.MODEL_DATA_DIR,
                               stage="scaled_fixed")   # -> model_data/batch_<span>/scaled_fixed/
assert scaled_export_fixed["matches_batch"].all(), \
    scaled_export_fixed.loc[~scaled_export_fixed["matches_batch"], ["table", "span"]].to_string()
msc.save_scaling_config({"types": {name: types for name, (_, types, _) in FRAMES.items()},
                         "heavy_ratios": HEAVY_RATIOS, "constants": SCALE_CONSTANTS,
                         "causal_window": CAUSAL_WINDOW, "causal_min_rows": CAUSAL_MIN_ROWS},
                        MODEL_DIR)
display(scaled_export_fixed)

wrote 4 tables -> model_data/batch_2024-04_2026-03/scaled_fixed/
 wrote scaling config -> model_data/batch_2024-04_2026-03/scaling_config.json


,table,file,rows,cols,first_date,last_date,span,matches_batch,superseded
0,DF_model_scaled_fixed_panel_2h,DF_model_scaled_fixed_panel_2h_2024-04-01_2026...,8736,46,2024-04-01,2026-03-31,2024-04 → 2026-03,True,
1,DF_model_scaled_fixed_market_2h,DF_model_scaled_fixed_market_2h_2024-04-01_202...,8736,26,2024-04-01,2026-03-31,2024-04 → 2026-03,True,
2,DF_model_scaled_fixed_liq_2h,DF_model_scaled_fixed_liq_2h_2024-04-01_2026-0...,8736,7,2024-04-01,2026-03-31,2024-04 → 2026-03,True,
3,DF_model_scaled_fixed_user_2h,DF_model_scaled_fixed_user_2h_2024-04-01_2026-...,8736,11,2024-04-01,2026-03-31,2024-04 → 2026-03,True,


In [21]:
SCALED_FRAMES_CASUAL = {
    **{f"DF_model_scaled_causal_{name}_2h": df for name, df in SCALED_CAUSAL.items()},
}
scaled_export_causal = run.save(SCALED_FRAMES_CASUAL, out_dir=cfg.MODEL_DATA_DIR,
                                stage="scaled_causal")  # -> model_data/batch_<span>/scaled_causal/
assert scaled_export_causal["matches_batch"].all(), \
    scaled_export_causal.loc[~scaled_export_causal["matches_batch"], ["table", "span"]].to_string()
# msc.save_scaling_config({"types": {name: types for name, (_, types, _) in FRAMES.items()},
#                          "heavy_ratios": HEAVY_RATIOS, "constants": SCALE_CONSTANTS,
#                          "causal_window": CAUSAL_WINDOW, "causal_min_rows": CAUSAL_MIN_ROWS},
#                         MODEL_DIR)
display(scaled_export_causal)

wrote 4 tables -> model_data/batch_2024-04_2026-03/scaled_causal/


,table,file,rows,cols,first_date,last_date,span,matches_batch,superseded
0,DF_model_scaled_causal_panel_2h,DF_model_scaled_causal_panel_2h_2024-04-01_202...,8736,46,2024-04-01,2026-03-31,2024-04 → 2026-03,True,
1,DF_model_scaled_causal_market_2h,DF_model_scaled_causal_market_2h_2024-04-01_20...,8736,26,2024-04-01,2026-03-31,2024-04 → 2026-03,True,
2,DF_model_scaled_causal_liq_2h,DF_model_scaled_causal_liq_2h_2024-04-01_2026-...,8736,7,2024-04-01,2026-03-31,2024-04 → 2026-03,True,
3,DF_model_scaled_causal_user_2h,DF_model_scaled_causal_user_2h_2024-04-01_2026...,8736,11,2024-04-01,2026-03-31,2024-04 → 2026-03,True,


## Targets — `DF_target_*` (kept apart from `DF_model_*`)

Four incoming columns, each rolled **forward** over `(t, t+h]` for h = 24h / 3d / 7d / 15d —
t itself is excluded, so a target never holds the row it is predicted from:

| target | forward rollup |
|---|---|
| `liquidation_debt_covered_value_usd` | sum |
| `net_liquidity_flow_usd` | sum (signed) |
| `avg_ltv` | mean weighted by `sampled_user_count` |
| `borrow_capacity_utilization` | ratio of sums, weighted by `sampled_user_count` |

These four columns (and their rolled copies) are **removed from every `df_*` / `DF_model_*`
frame** above (`TARGET_BASES`, cell 5); they exist only here.

Then scaled with the **same fixed scaling** as the features (no min-max — it could not be
reversed); a prediction is turned back into USD / a ratio by `msc.unscale_targets`. The signed
target uses the same asinh shape without the ±reference clip, so it reverses exactly.

Rows whose window runs past the data are left **blank (pending)** — the last 24h / 3d / 7d / 15d.
They are the prediction rows: predicted now, scored when the next market data arrives and the
targets are rebuilt over old + new data. Features and targets meet only at train / val / test
time, on `time_bucket`, one target per model (`mds.join_for_training`, `mds.prediction_rows`).
Plumbing: `src/model_dataset.py` (forward block), `src/model_scaling.py` (reverse block).

In [22]:
# --- target config: incoming columns only, scaled by the feature categories + SCALE_CONSTANTS ---
TARGET_WINDOWS = ('24h', '3d', '7d')          # ('24h', '3d', '7d', '15d')
TARGET_SUMS  = ("liquidation_debt_covered_value_usd", "net_liquidity_flow_usd")
TARGET_MEANS = ("avg_ltv",)                    # weighted by USER_WEIGHT
TARGET_RATIOS = {                              # sampled users' debt / their borrowing limit
    "borrow_capacity_utilization": ("avg_total_debt_base",
                                    ("avg_total_debt_base", "avg_available_borrows_base"))}
assert set((*TARGET_SUMS, *TARGET_MEANS, *TARGET_RATIOS)) == set(TARGET_BASES)
TARGET_TYPES = {
    "amount":        ("liquidation_debt_covered_value_usd",),
    "signed_amount": ("net_liquidity_flow_usd",),    # scaled without the ±reference clip
    "share":         ("avg_ltv", "borrow_capacity_utilization"),
}
TARGET_DATA_DIR = "target_data"                # own folder: never mixed with model_data/

In [23]:
# targets: the incoming 2h columns on the panel's grid, independent of the columns dropped above,
# rolled FORWARD over (t, t+h]. A pending row (window past the data) stays NaN = a prediction row.
TARGET_GRID = MAIN[["time_bucket", "net_liquidity_flow_usd"]]
df_target_src = mds.align_to_grid(
    run.load(SOURCES["liq_2h"])[["time_bucket", "liquidation_debt_covered_value_usd"]],
    TARGET_GRID, fill=0)                                   # no liquidation row = 0
df_target_src = mds.align_to_grid(
    run.load(SOURCES["user_2h"])[["time_bucket", "avg_ltv", "avg_total_debt_base",
                                  "avg_available_borrows_base", USER_WEIGHT]],
    df_target_src, fill={USER_WEIGHT: 0})                  # unsampled bucket = weight 0, avg_* NaN

df_target = mds.forward_sums(df_target_src, TARGET_SUMS, window=TARGET_WINDOWS)
df_target = mds.forward_means(df_target, TARGET_MEANS, window=TARGET_WINDOWS, weight=USER_WEIGHT)
df_target = mds.forward_ratio_of_sums(df_target, TARGET_RATIOS, window=TARGET_WINDOWS,
                                      weight=USER_WEIGHT)
# time_bucket + the forward columns only: nothing from the feature side travels with the targets
DF_target_raw = df_target[["time_bucket", *[c for c in df_target.columns if "_fwd_" in c]]]
# joined batches: a forward window across a hole in the grid (the seam) would read missing
# buckets as "nothing happened" -> those labels are blanked (unobserved). Single batch: no gaps.
GAPS = mds.grid_gaps(TARGET_GRID)
display(GAPS)
DF_target_raw = mds.blank_across_gaps(DF_target_raw, GAPS)
print(f" {DF_target_raw.shape[0]} rows x {DF_target_raw.shape[1]} cols")
display(mds.target_status(DF_target_raw))   # labelled / pending (to predict) / unobserved (no sample)

 aligned 2278 rows onto a 8736-row grid (6458 bucket(s) filled)
 aligned 3803 rows onto a 8736-row grid (4933 bucket(s) filled)
 forward ['24h', '3d', '7d'] sums added for 2 column(s) (6 columns)
 forward ['24h', '3d', '7d'] means added for 1 column(s) (3 columns), weighted by sampled_user_count
 1 ratio(s) rebuilt as forward ratio-of-sums over ['24h', '3d', '7d'] (3 columns), weighted by sampled_user_count
 1 gap(s) in the 0 days 02:00:00 grid, 12 bucket(s) missing


,last_before,first_after,missing_buckets
0,2025-03-31 00:00:00+00:00,2025-04-01 02:00:00+00:00,12


 blanked targets whose window crosses a gap: {'liquidation_debt_covered_value_usd_fwd_24h': 11, 'liquidation_debt_covered_value_usd_fwd_3d': 36, 'liquidation_debt_covered_value_usd_fwd_7d': 84, 'net_liquidity_flow_usd_fwd_24h': 11, 'net_liquidity_flow_usd_fwd_3d': 36, 'net_liquidity_flow_usd_fwd_7d': 84, 'avg_ltv_fwd_24h': 11, 'avg_ltv_fwd_3d': 36, 'avg_ltv_fwd_7d': 84, 'borrow_capacity_utilization_fwd_24h': 11, 'borrow_capacity_utilization_fwd_3d': 36, 'borrow_capacity_utilization_fwd_7d': 84}
 8736 rows x 13 cols


,column,window,labelled,pending,unobserved,first_pending
0,liquidation_debt_covered_value_usd_fwd_24h,24h,8712,12,12,2026-03-30 02:00:00+00:00
1,liquidation_debt_covered_value_usd_fwd_3d,3d,8664,36,36,2026-03-28 02:00:00+00:00
2,liquidation_debt_covered_value_usd_fwd_7d,7d,8568,84,84,2026-03-24 02:00:00+00:00
3,net_liquidity_flow_usd_fwd_24h,24h,8712,12,12,2026-03-30 02:00:00+00:00
4,net_liquidity_flow_usd_fwd_3d,3d,8664,36,36,2026-03-28 02:00:00+00:00
5,net_liquidity_flow_usd_fwd_7d,7d,8568,84,84,2026-03-24 02:00:00+00:00
6,avg_ltv_fwd_24h,24h,8559,12,165,2026-03-30 02:00:00+00:00
7,avg_ltv_fwd_3d,3d,8664,36,36,2026-03-28 02:00:00+00:00
8,avg_ltv_fwd_7d,7d,8568,84,84,2026-03-24 02:00:00+00:00
9,borrow_capacity_utilization_fwd_24h,24h,8549,12,175,2026-03-30 02:00:00+00:00


In [24]:
# target profile (info only): null %, zero %, spread per target
temp_2 = adv.statistical_validation(DF_target_raw, columns=[c for c in DF_target_raw.columns
                                                            if c != "time_bucket"], save=False)
display(temp_2[["column", "null_pct", "zero_pct", "mean", "cv", "p95"]])

,column,null_pct,zero_pct,mean,cv,p95
0,liquidation_debt_covered_value_usd_fwd_24h,0.2747,11.1341,2.478769e+06,6.055776,6.312438e+06
1,liquidation_debt_covered_value_usd_fwd_3d,0.8242,0.6233,7.453397e+06,3.761647,3.798505e+07
2,liquidation_debt_covered_value_usd_fwd_7d,1.9231,0.0000,1.739386e+07,2.665500,1.085871e+08
3,net_liquidity_flow_usd_fwd_24h,0.2747,0.0000,4.376705e+07,7.185678,5.355371e+08
4,net_liquidity_flow_usd_fwd_3d,0.8242,0.0000,1.336286e+08,3.989806,1.059336e+09
5,net_liquidity_flow_usd_fwd_7d,1.9231,0.0000,3.137765e+08,2.528679,1.653469e+09
6,avg_ltv_fwd_24h,2.0261,0.1168,7.903549e-01,0.110271,9.050000e-01
7,avg_ltv_fwd_3d,0.8242,0.0000,7.873364e-01,0.065904,8.649130e-01
8,avg_ltv_fwd_7d,1.9231,0.0000,7.858763e-01,0.044140,8.392952e-01
9,borrow_capacity_utilization_fwd_24h,2.1406,0.5264,7.494182e-01,0.262941,9.724016e-01


In [25]:
# # profile check: DF_target_raw with the last 15d (the pending tail) excluded
# EMBARGO_ROWS = mds.window_rows(DF_target_raw, "15d")        # 180 at 2h grain — the longest horizon
# DF_target_check = DF_target_raw.iloc[:-EMBARGO_ROWS]
# print(f" {DF_target_check.shape[0]} rows (of {DF_target_raw.shape[0]}, last {EMBARGO_ROWS} dropped)")

# temp_check = adv.statistical_validation(
#     DF_target_check, columns=[c for c in DF_target_check.columns if c != "time_bucket"],
#     save=False)
# display(temp_check[["column", "null_pct", "zero_pct", "mean", "cv", "p95"]])

In [26]:
# scale with the SAME fixed scaling as the features; every mapping used here has a reverse
TARGET_COLMAP = msc.target_colmap(DF_target_raw, TARGET_TYPES, window=TARGET_WINDOWS)
display(msc.category_table(TARGET_COLMAP))
DF_target_scaled = msc.scale_targets(DF_target_raw, TARGET_COLMAP, SCALE_CONSTANTS)

# reverse check: unscale(scale(x)) must give x back
round_trip = msc.round_trip_report(DF_target_raw, TARGET_COLMAP, SCALE_CONSTANTS)
display(round_trip)
assert (round_trip["max_rel_err"] < 1e-9).all() and round_trip["nan_same"].all(), round_trip

# separation: no target column (forward, base or rolled copy) may sit in any feature frame
for name, frame in {**MODEL_FRAMES, **SCALED_FRAMES_FIXED, **SCALED_FRAMES_CASUAL}.items():
    leaked = [c for c in frame.columns if "_fwd_" in c
              or any(c == b or c.startswith(f"{b}_") for b in TARGET_BASES)]
    assert not leaked, f"{name} carries target column(s) {leaked}"
DF_target_scaled

,column,category,source,shape,base,fixed,rolled,group
0,liquidation_debt_covered_value_usd_fwd_24h,amount,liquidation_debt_covered_value_usd,sqrt_log,100.0,ceiling,True,None
1,liquidation_debt_covered_value_usd_fwd_3d,amount,liquidation_debt_covered_value_usd,sqrt_log,100.0,ceiling,True,None
2,liquidation_debt_covered_value_usd_fwd_7d,amount,liquidation_debt_covered_value_usd,sqrt_log,100.0,ceiling,True,None
3,avg_ltv_fwd_24h,share,avg_ltv,identity,NaN,percent,True,None
4,avg_ltv_fwd_3d,share,avg_ltv,identity,NaN,percent,True,None
5,avg_ltv_fwd_7d,share,avg_ltv,identity,NaN,percent,True,None
6,borrow_capacity_utilization_fwd_24h,share,borrow_capacity_utilization,identity,NaN,percent,True,None
7,borrow_capacity_utilization_fwd_3d,share,borrow_capacity_utilization,identity,NaN,percent,True,None
8,borrow_capacity_utilization_fwd_7d,share,borrow_capacity_utilization,identity,NaN,percent,True,None
9,net_liquidity_flow_usd_fwd_24h,signed_amount,net_liquidity_flow_usd,asinh,NaN,symmetric_open,True,None


 fixed scaling: 12 column(s), no statistic of the data used
 fixed scaling: 12 column(s), no statistic of the data used


,column,fixed,max_abs_err,max_rel_err,nan_same
0,liquidation_debt_covered_value_usd_fwd_24h,ceiling,1.043081e-06,6.956546e-15,True
1,liquidation_debt_covered_value_usd_fwd_3d,ceiling,1.996756e-06,8.396713e-15,True
2,liquidation_debt_covered_value_usd_fwd_7d,ceiling,2.115965e-06,8.708102e-15,True
3,net_liquidity_flow_usd_fwd_24h,symmetric_open,3.814697e-06,1.038165e-13,True
4,net_liquidity_flow_usd_fwd_3d,symmetric_open,6.198883e-06,1.590026e-13,True
5,net_liquidity_flow_usd_fwd_7d,symmetric_open,1.049042e-05,5.160597e-15,True
6,avg_ltv_fwd_24h,percent,1.110223e-16,1.110223e-16,True
7,avg_ltv_fwd_3d,percent,1.110223e-16,1.110223e-16,True
8,avg_ltv_fwd_7d,percent,1.110223e-16,1.110223e-16,True
9,borrow_capacity_utilization_fwd_24h,percent,1.110223e-16,1.110223e-16,True


,time_bucket,liquidation_debt_covered_value_usd_fwd_24h,liquidation_debt_covered_value_usd_fwd_3d,liquidation_debt_covered_value_usd_fwd_7d,net_liquidity_flow_usd_fwd_24h,net_liquidity_flow_usd_fwd_3d,net_liquidity_flow_usd_fwd_7d,avg_ltv_fwd_24h,avg_ltv_fwd_3d,avg_ltv_fwd_7d,borrow_capacity_utilization_fwd_24h,borrow_capacity_utilization_fwd_3d,borrow_capacity_utilization_fwd_7d
0,2024-04-01 02:00:00.000 UTC,45.598164,62.927677,62.975148,27.780134,24.446061,76.507136,82.112222,79.818333,77.793929,80.306027,97.987487,96.876826
1,2024-04-01 04:00:00.000 UTC,56.508327,62.927677,62.975148,23.845173,22.258420,75.021265,80.871333,79.818333,77.793929,79.720250,97.987487,96.876826
2,2024-04-01 06:00:00.000 UTC,56.500988,62.926252,62.973272,34.458676,26.962468,78.814371,79.523704,79.042941,77.230741,77.791196,97.994742,96.874057
3,2024-04-01 08:00:00.000 UTC,56.500988,62.926252,62.973601,66.562561,62.950727,80.008402,79.523704,79.042941,78.318276,77.791196,97.994742,98.344483
4,2024-04-01 10:00:00.000 UTC,56.500988,62.926252,62.976360,69.594947,70.398393,80.433093,79.523704,79.042941,78.750882,77.791196,97.994742,97.801226
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8731,2026-03-30 16:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8732,2026-03-30 18:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8733,2026-03-30 20:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8734,2026-03-30 22:00:00.000 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
# --- export: raw + scaled targets and the config that reverses them -> target_data/batch_<span>/ ---
TARGET_FRAMES = {"DF_target_raw_2h": DF_target_raw, "DF_target_scaled_2h": DF_target_scaled}
target_export = run.save(TARGET_FRAMES, out_dir=TARGET_DATA_DIR)
assert target_export["matches_batch"].all(), \
    target_export.loc[~target_export["matches_batch"], ["table", "span"]].to_string()
msc.save_scaling_config({"types": TARGET_TYPES, "windows": TARGET_WINDOWS,
                         "constants": SCALE_CONSTANTS,
                         "map_swap": msc.TARGET_MAP_SWAP},
                        run.folder(TARGET_DATA_DIR), name="target_scaling_config.json")
display(target_export)

wrote 2 tables -> target_data/batch_2024-04_2026-03/
 wrote scaling config -> target_data/batch_2024-04_2026-03/target_scaling_config.json


,table,file,rows,cols,first_date,last_date,span,matches_batch,superseded
0,DF_target_raw_2h,DF_target_raw_2h_2024-04-01_2026-03-31.csv,8736,13,2024-04-01,2026-03-31,2024-04 → 2026-03,True,
1,DF_target_scaled_2h,DF_target_scaled_2h_2024-04-01_2026-03-31.csv,8736,13,2024-04-01,2026-03-31,2024-04 → 2026-03,True,
